In [1]:
from pathlib import Path

import pandas as pd

from dotenv import load_dotenv

from langchain_core.documents import Document
from langchain_core.tools import tool

from langchain_text_splitters import (
    RecursiveCharacterTextSplitter
)

from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

from langchain_groq import ChatGroq

from langgraph.prebuilt import create_react_agent

/Users/aaravmenon/New folder/Attendance RAG/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
load_dotenv()

print("Environment loaded.")

Environment loaded.


In [3]:
data_path = Path("../data/structured")

employees = pd.read_csv(
    data_path / "employees.csv"
)

attendance = pd.read_csv(
    data_path / "attendance.csv"
)

leave = pd.read_csv(
    data_path / "leave.csv"
)

holidays = pd.read_csv(
    data_path / "holidays.csv"
)

attendance["date"] = pd.to_datetime(
    attendance["date"]
)

leave["start_date"] = pd.to_datetime(
    leave["start_date"]
)

leave["end_date"] = pd.to_datetime(
    leave["end_date"]
)

holidays["date"] = pd.to_datetime(
    holidays["date"]
)

print("Structured data loaded.")
print(f"Employees: {len(employees)}")
print(f"Attendance records: {len(attendance)}")
print(f"Leave records: {len(leave)}")

Structured data loaded.
Employees: 250
Attendance records: 41750
Leave records: 1550


In [4]:
kb_path = Path("../data/knowledge_base")

documents = []

for file in kb_path.glob("*.md"):

    text = file.read_text(
        encoding="utf-8"
    )

    documents.append(
        Document(
            page_content=text,
            metadata={
                "source": file.name
            }
        )
    )

print(
    f"Loaded {len(documents)} policy documents."
)

Loaded 6 policy documents.


In [5]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=700,
    chunk_overlap=150,
    separators=[
        "\n\n",
        "\n",
        ".",
        " "
    ]
)

chunks = splitter.split_documents(
    documents
)

print(
    f"Created {len(chunks)} policy chunks."
)

Created 8 policy chunks.


In [6]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5438.17it/s]


Embedding model loaded.


In [7]:
vector_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="nexatech_final_policies"
)

print(
    "Vectors stored:",
    vector_store._collection.count()
)

Vectors stored: 8


In [8]:
retriever = vector_store.as_retriever(
    search_kwargs={
        "k": 4
    }
)

print("Retriever ready.")

Retriever ready.


In [9]:
@tool
def search_company_policy(
    question: str
) -> str:
    """
    Search company policy documents.

    Use this for questions about company rules,
    attendance requirements, remote work,
    working hours, leave policies, overtime,
    holidays, and workplace regulations.

    Do not use this for an employee's actual
    attendance records.
    """

    retrieved_docs = retriever.invoke(
        question
    )

    if not retrieved_docs:
        return "No relevant company policy was found."

    results = []

    for doc in retrieved_docs:

        results.append(
            f"Source: {doc.metadata['source']}\n"
            f"{doc.page_content}"
        )

    return "\n\n---\n\n".join(results)

In [10]:
@tool
def get_employee(
    employee_id: str
) -> str:
    """
    Get employee information such as name,
    department, role, employment type,
    weekly hours, vacation entitlement,
    and office location.
    """

    result = employees[
        employees["employee_id"] == employee_id
    ]

    if result.empty:
        return (
            f"No employee found with ID "
            f"{employee_id}."
        )

    employee = result.iloc[0]

    return (
        f"Employee ID: {employee['employee_id']}\n"
        f"Name: {employee['name']}\n"
        f"Department: {employee['department']}\n"
        f"Role: {employee['role']}\n"
        f"Employment type: {employee['employment_type']}\n"
        f"Weekly hours: {employee['weekly_hours']}\n"
        f"Vacation entitlement: {employee['vacation_days']} days\n"
        f"Office location: {employee['office_location']}"
    )

In [11]:
@tool
def get_attendance_summary(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Get actual attendance statistics for an employee
    during a specified date range.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ]

    if records.empty:
        return (
            f"No attendance records found for "
            f"{employee_id}."
        )

    office_days = (
        records["location"] == "office"
    ).sum()

    home_days = (
        records["location"] == "home"
    ).sum()

    business_trip_days = (
        records["location"] == "business_trip"
    ).sum()

    sick_days = (
        records["status"] == "sick"
    ).sum()

    leave_days = (
        records["status"] == "leave"
    ).sum()

    missing_records = records[
        records["status"].isin(
            ["missing", "missing_checkout"]
        )
    ].shape[0]

    return (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Office days: {office_days}\n"
        f"Home-office days: {home_days}\n"
        f"Business-trip days: {business_trip_days}\n"
        f"Sick days: {sick_days}\n"
        f"Leave days: {leave_days}\n"
        f"Missing attendance records: {missing_records}"
    )

In [12]:
@tool
def calculate_working_hours(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Calculate total recorded working hours
    during a specified date range.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ].copy()

    records = records[
        records["check_in"].notna() &
        records["check_out"].notna() &
        (records["check_in"] != "") &
        (records["check_out"] != "")
    ]

    if records.empty:
        return (
            f"No complete attendance records "
            f"found for {employee_id}."
        )

    records["check_in_time"] = pd.to_datetime(
        records["check_in"],
        format="%H:%M"
    )

    records["check_out_time"] = pd.to_datetime(
        records["check_out"],
        format="%H:%M"
    )

    records["hours"] = (
        records["check_out_time"]
        - records["check_in_time"]
    ).dt.total_seconds() / 3600

    total_hours = records["hours"].sum()

    return (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Total recorded working hours: "
        f"{total_hours:.2f}"
    )

In [13]:
@tool
def get_leave_balance(
    employee_id: str
) -> str:
    """
    Get an employee's vacation entitlement,
    approved vacation used, and remaining vacation.
    """

    employee_result = employees[
        employees["employee_id"] == employee_id
    ]

    if employee_result.empty:
        return (
            f"No employee found with ID "
            f"{employee_id}."
        )

    employee = employee_result.iloc[0]

    entitlement = int(
        employee["vacation_days"]
    )

    approved_vacation = leave[
        (leave["employee_id"] == employee_id) &
        (leave["type"] == "vacation") &
        (leave["status"] == "approved")
    ]

    used = int(
        approved_vacation["days"].sum()
    )

    remaining = entitlement - used

    return (
        f"Employee: {employee_id}\n"
        f"Vacation entitlement: {entitlement} days\n"
        f"Approved vacation used: {used} days\n"
        f"Remaining vacation: {remaining} days"
    )

In [14]:
@tool
def find_missing_attendance(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Find missing attendance records and
    missing check-outs.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ]

    missing = records[
        records["status"].isin(
            ["missing", "missing_checkout"]
        )
    ]

    if missing.empty:
        return (
            f"No missing attendance records found "
            f"for {employee_id}."
        )

    results = []

    for _, row in missing.iterrows():

        results.append(
            f"{row['date'].date()} - "
            f"{row['status']}"
        )

    return (
        f"Missing attendance for {employee_id}:\n"
        + "\n".join(results)
    )

In [15]:
POLICY_CONFIG = {
    "minimum_office_days_per_week": 3
}

In [16]:
def calculate_weekly_attendance(
    employee_id,
    start_date,
    end_date
):
    """
    Calculate weekly office and remote attendance.
    """

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ].copy()

    if records.empty:
        return pd.DataFrame()

    records["week"] = (
        records["date"]
        .dt.to_period("W-MON")
        .astype(str)
    )

    weekly = (
        records
        .groupby("week")
        .agg(
            office_days=(
                "location",
                lambda x: (x == "office").sum()
            ),
            home_days=(
                "location",
                lambda x: (x == "home").sum()
            )
        )
        .reset_index()
    )

    return weekly

In [17]:
@tool
def check_remote_work_compliance(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Determine whether an employee met the company's
    weekly office attendance requirement.
    """

    weekly = calculate_weekly_attendance(
        employee_id,
        start_date,
        end_date
    )

    if weekly.empty:
        return (
            f"No attendance data found for "
            f"{employee_id}."
        )

    required = POLICY_CONFIG[
        "minimum_office_days_per_week"
    ]

    weekly["compliant"] = (
        weekly["office_days"] >= required
    )

    failed = weekly[
        ~weekly["compliant"]
    ]

    overall = bool(
        weekly["compliant"].all()
    )

    status = (
        "COMPLIANT"
        if overall
        else "NOT COMPLIANT"
    )

    result = (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Required office days per week: {required}\n"
        f"Weeks checked: {len(weekly)}\n"
        f"Compliant weeks: "
        f"{weekly['compliant'].sum()}\n"
        f"Non-compliant weeks: "
        f"{(~weekly['compliant']).sum()}\n"
        f"Overall status: {status}"
    )

    if not failed.empty:

        result += (
            "\n\nWeeks below requirement:\n"
        )

        for _, row in failed.iterrows():

            result += (
                f"- {row['week']}: "
                f"{row['office_days']} office days\n"
            )

    return result

In [18]:
@tool
def check_working_hours_compliance(
    employee_id: str,
    start_date: str,
    end_date: str
) -> str:
    """
    Compare recorded working hours against
    the employee's contracted working hours.
    """

    employee_result = employees[
        employees["employee_id"] == employee_id
    ]

    if employee_result.empty:
        return (
            f"No employee found with ID "
            f"{employee_id}."
        )

    employee = employee_result.iloc[0]

    weekly_hours = float(
        employee["weekly_hours"]
    )

    start = pd.to_datetime(start_date)
    end = pd.to_datetime(end_date)

    records = attendance[
        (attendance["employee_id"] == employee_id) &
        (attendance["date"] >= start) &
        (attendance["date"] <= end)
    ].copy()

    records = records[
        records["check_in"].notna() &
        records["check_out"].notna() &
        (records["check_in"] != "") &
        (records["check_out"] != "")
    ]

    if records.empty:
        return (
            "No complete attendance records "
            "were found."
        )

    records["check_in_time"] = pd.to_datetime(
        records["check_in"],
        format="%H:%M"
    )

    records["check_out_time"] = pd.to_datetime(
        records["check_out"],
        format="%H:%M"
    )

    records["hours"] = (
        records["check_out_time"]
        - records["check_in_time"]
    ).dt.total_seconds() / 3600

    recorded_hours = records["hours"].sum()

    days = (end - start).days + 1

    expected_hours = (
        weekly_hours * days / 7
    )

    status = (
        "COMPLIANT"
        if recorded_hours >= expected_hours
        else "BELOW EXPECTED HOURS"
    )

    return (
        f"Employee: {employee_id}\n"
        f"Period: {start_date} to {end_date}\n"
        f"Contracted weekly hours: {weekly_hours:.2f}\n"
        f"Recorded hours: {recorded_hours:.2f}\n"
        f"Expected hours: {expected_hours:.2f}\n"
        f"Status: {status}"
    )

In [19]:
tools = [
    search_company_policy,
    get_employee,
    get_attendance_summary,
    calculate_working_hours,
    find_missing_attendance,
    get_leave_balance,
    check_remote_work_compliance,
    check_working_hours_compliance
]

print("Agent tools:")

for tool in tools:
    print("-", tool.name)

Agent tools:
- search_company_policy
- get_employee
- get_attendance_summary
- calculate_working_hours
- find_missing_attendance
- get_leave_balance
- check_remote_work_compliance
- check_working_hours_compliance


In [20]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0,
    reasoning_format="parsed"
)

print("LLM ready.")

LLM ready.


In [21]:
agent = create_react_agent(
    model=llm,
    tools=tools
)

print("Final agent created successfully.")

Final agent created successfully.


/var/folders/kb/3rhgzw5d6md74w5kr02cc4cm0000gn/T/ipykernel_2286/1339849759.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(


In [22]:
question = """
How many days did E0001 work from home
in August 2026?
"""

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

E0001 worked from home **5 days** in August 2026.


In [23]:
question = """
What is the company's policy regarding
working from home?
"""

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

**NexaTech GmbH – Working‑from‑Home (Remote‑Work) Policy**

| Aspect | What the policy says |
|--------|----------------------|
| **Eligibility** | Full‑time employees may work remotely if their role is allowed to do so. |
| **Maximum remote days** | Up to **2 regular working days per week** can be taken as remote work. |
| **Office‑attendance requirement** | Employees on a hybrid schedule must be in the office **at least 3 days per week** (remote days do **not** count toward this minimum). |
| **Core hours while remote** | You must be reachable and available **09:00 – 15:00** on remote‑work days. |
| **Start/finish flexibility** | You may start between **07:00 – 09:00** and finish accordingly, as long as you meet your contracted weekly hours. |
| **Exceptions** | Managers can grant temporary exceptions for business, family, or other justified reasons. |
| **Attendance recording** | Remote‑work days are recorded as normal working days in the attendance system; they do not replace the r

In [24]:
question = """
Did E0001 comply with the company's
office attendance requirement in August 2026?
"""

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

**Result:** Employee **E0001** did **not** meet the company’s office‑attendance requirement for August 2026.

**Details**

| Week (Mon–Sun) | Office days recorded | Requirement (days) | Status |
|----------------|----------------------|--------------------|--------|
| 2026‑07‑28 – 2026‑08‑03 | 1 | 3 | **Non‑compliant** |
| 2026‑08‑04 – 2026‑08‑10 | 3 | 3 | Compliant |
| 2026‑08‑11 – 2026‑08‑17 | 3 | 3 | Compliant |
| 2026‑08‑18 – 2026‑08‑24 | 3 | 3 | Compliant |
| 2026‑08‑25 – 2026‑08‑31 | 3 | 3 | Compliant |

- **Total weeks checked:** 5  
- **Compliant weeks:** 4  
- **Non‑compliant weeks:** 1  

Because the employee fell short of the required **3 office days** in the week of **July 28 – August 3**, the overall status for the month is **NOT COMPLIANT**.

If you need a deeper dive (e.g., specific dates of attendance, reasons for the shortfall, or next‑step recommendations), just let me know!


In [26]:
question = """
How many days did E0001 work from home in August 2026,
what does the company policy allow, and did
E0001 comply with the policy?
"""

response = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": question
            }
        ]
    }
)

print(
    response["messages"][-1].content
)

**Home‑office days in August 2026**  
- **E0001 recorded 5 home‑office days** (see the attendance summary for 2026‑08‑01 → 2026‑08‑31).

**What the company policy permits**  
- The NexaTech Remote‑Work Policy states that **full‑time employees may work remotely for a maximum of two regular working days per week**.  
- Consequently, hybrid employees must be present in the office **at least three days each week** (the “Office Requirement” in the Attendance Policy).

**Compliance check**  
- The system‑generated compliance check (`check_remote_work_compliance`) examined each calendar week in August 2026.  
- Out of the five weeks in the period, **four weeks met the three‑day‑in‑office minimum**, but **one week (2026‑07‑28 → 2026‑08‑03) did not** – E0001 was in the office only once that week.  
- Because the weekly office‑attendance requirement was not satisfied for that week, **E0001 is classified as “NOT COMPLIANT” with the remote‑work policy for August 2026**.

**Summary**

| Item | Deta